# Credit Default Prediction — XGBoost Baseline, Tuning, Calibration & Final Model

This notebook trains an XGBoost classifier to predict `default`, tunes
hyperparameters, checks whether probability calibration helps, tries early
stopping, and finally refits the best-performing configuration on the full
training set to score `test.csv`.

In [3]:
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
)
from sklearn.metrics import log_loss
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from scipy.stats import uniform, randint

RANDOM_STATE = 42

## 1. Load data and split

In [4]:
train = pd.read_csv('train.csv')

feature_cols = [c for c in train.columns if c not in ('client_id', 'default')]

X = train[feature_cols]
y = train['default']

# NOTE: variable names are kept consistent as X_train/X_val/y_train/y_val
# throughout the whole notebook (the original had a X_valid/y_valid vs
# X_val/y_val mismatch that caused a NameError when cells were run in order).
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

## 2. Baseline model

In [5]:
xg_model = xgb.XGBClassifier(
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    n_jobs=-1,
)

xg_model.fit(X_train, y_train)

val_probs = xg_model.predict_proba(X_val)[:, 1]
baseline_log_loss = log_loss(y_val, val_probs)
print("Baseline log loss:", baseline_log_loss)

Baseline log loss: 0.46355642663106733


## 3. Hyperparameter tuning (RandomizedSearchCV)

In [6]:
param_dist = {
    'n_estimators': randint(100, 600),
    'max_depth': randint(3, 8),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 10),
    'gamma': uniform(0, 5),
}

xg_search = RandomizedSearchCV(
    xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_distributions=param_dist,
    n_iter=30,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='neg_log_loss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

xg_search.fit(X_train, y_train)

print("Best params:", xg_search.best_params_)
print("Best CV log loss:", -xg_search.best_score_)

best_xg_model = xg_search.best_estimator_

# Validation-set score of the tuned model (uncalibrated)
tuned_val_probs = best_xg_model.predict_proba(X_val)[:, 1]
tuned_log_loss = log_loss(y_val, tuned_val_probs)
print("Tuned model validation log loss:", tuned_log_loss)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params: {'colsample_bytree': np.float64(0.9022204554172195), 'gamma': np.float64(1.1439908274581123), 'learning_rate': np.float64(0.0330939729486379), 'max_depth': 5, 'min_child_weight': 3, 'n_estimators': 185, 'subsample': np.float64(0.9521871356061031)}
Best CV log loss: 0.4249758578060939
Tuned model validation log loss: 0.43496359954049446


## 4. Calibration check

Let's see whether calibrating the tuned model's probabilities (isotonic
regression) improves validation log loss versus leaving it uncalibrated.

In [7]:
calibrated_xg_model = CalibratedClassifierCV(best_xg_model, method='isotonic', cv=5)
calibrated_xg_model.fit(X_train, y_train)

cal_val_probs = calibrated_xg_model.predict_proba(X_val)[:, 1]
calibrated_log_loss = log_loss(y_val, cal_val_probs)
print("Calibrated validation log loss:  ", calibrated_log_loss)
print("Uncalibrated validation log loss:", tuned_log_loss)

Calibrated validation log loss:   0.4362216480795393
Uncalibrated validation log loss: 0.43496359954049446


**Decision:** the uncalibrated tuned model has the better (lower) log loss,
so we keep it uncalibrated and move on.

## 5. Early stopping

Instead of using the `n_estimators` value chosen by the random search, let's
give the model a generous cap on trees and let early stopping pick the
actual number of boosting rounds based on validation performance.

In [8]:
# Take the tuned hyperparameters but drop n_estimators — early stopping
# will decide how many trees to use instead.
best_xg_params = dict(xg_search.best_params_)
best_xg_params.pop('n_estimators', None)

final_xg_model = xgb.XGBClassifier(
    **best_xg_params,
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=1000,        # high cap; early stopping will cut it short
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

final_xg_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

print("Best iteration:", final_xg_model.best_iteration)

Best iteration: 181


In [9]:
final_val_probs = final_xg_model.predict_proba(X_val)[:, 1]
final_log_loss = log_loss(y_val, final_val_probs)
print("Early-stopped model validation log loss:", final_log_loss)

Early-stopped model validation log loss: 0.4348797718842057


## 6. Summary of validation results

In [10]:
results = pd.DataFrame({
    'model': ['Baseline', 'Tuned (uncalibrated)', 'Tuned + isotonic calibration', 'Tuned + early stopping'],
    'val_log_loss': [baseline_log_loss, tuned_log_loss, calibrated_log_loss, final_log_loss],
}).sort_values('val_log_loss').reset_index(drop=True)

results

,model,val_log_loss
0,Tuned + early stopping,0.434880
1,Tuned (uncalibrated),0.434964
2,Tuned + isotonic calibration,0.436222
3,Baseline,0.463556


Based on this table, `final_xg_model` (tuned hyperparameters + early
stopping) has the best validation log loss, so it's the configuration we
carry forward to fit on the full dataset and score `test.csv`.

**Why not just call `final_xg_model.fit(X, y)` directly?** Early stopping
needs a held-out validation set to know when to stop — if we fit on 100% of
the data there's nothing left to validate against. Instead we fix
`n_estimators` to the `best_iteration` found above and retrain a plain
(non-early-stopping) model with that fixed tree count on all of the data.

## 7. Refit best configuration on the full training set and predict on test.csv

In [11]:
# Fix n_estimators at the iteration count early stopping settled on
# (+1 because best_iteration is 0-indexed).
full_fit_params = dict(best_xg_params)
full_fit_params['n_estimators'] = final_xg_model.best_iteration + 1

production_model = xgb.XGBClassifier(
    **full_fit_params,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# Refit on ALL of train.csv (X, y), not just the train split
production_model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=np.float64(0.9022204554172195), device=None,
              early_stopping_rounds=None, enable_categorical=True,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=np.float64(1.1439908274581123), grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=np.float64(0.0330939729486379), max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=182, n_jobs=-1,
              num_parallel_tree=None, ...)

In [12]:
test_df = pd.read_csv('test.csv')

test_probs = production_model.predict_proba(test_df[feature_cols])[:, 1]
test_probs = np.clip(test_probs, 1e-6, 1 - 1e-6)  # safety clip

prediction = pd.DataFrame({
    'client_id': test_df['client_id'],
    'default': test_probs,
})
prediction.to_csv('prediction.csv', index=False)

prediction.head()

,client_id,default
0,CC_0012A082B7B7,0.139346
1,CC_0012BC27DFB6,0.161177
2,CC_001563B2143D,0.100567
3,CC_001B46930B6F,0.092672
4,CC_001D771E1C9F,0.079607


## 8. Calibrated version of the final model

The comparison in Section 4 showed calibration hurt performance on that
particular train/val split, but you may still want calibrated probabilities
for downstream use (e.g. if probabilities feed into a cost/risk
calculation where well-calibrated outputs matter more than raw log loss).
This section refits a calibrated model on the full training set and writes
its predictions to a separate CSV so you can compare both submissions.

We use the same tuned hyperparameters and fixed `n_estimators` (from early
stopping) as `production_model`, but wrap the estimator in
`CalibratedClassifierCV` with `cv=5` — this internally cross-validates on
`X`/`y` to fit the calibrator, so no separate held-out split is needed.

In [13]:
calibrated_production_model = CalibratedClassifierCV(
    xgb.XGBClassifier(
        **full_fit_params,
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    method='isotonic',
    cv=5,
)

# Fit on ALL of train.csv, same as production_model
calibrated_production_model.fit(X, y)

CalibratedClassifierCV(cv=5,
                       estimator=XGBClassifier(base_score=None, booster=None,
                                               callbacks=None,
                                               colsample_bylevel=None,
                                               colsample_bynode=None,
                                               colsample_bytree=np.float64(0.9022204554172195),
                                               device=None,
                                               early_stopping_rounds=None,
                                               enable_categorical=True,
                                               eval_metric='logloss',
                                               feature_types=None,
                                               feature_weights=None,
                                               gamma=np.float64(1.1439908274581123)...
                                               importance_type=None,
                                               interaction_constraints=None,
                                               learning_rate=np.float64(0.0330939729486379),
                                               max_bin=None,
                                               max_cat_threshold=None,
                                               max_cat_to_onehot=None,
                                               max_delta_step=None, max_depth=5,
                                               max_leaves=None,
                                               min_child_weight=3, missing=nan,
                                               monotone_constraints=None,
                                               multi_strategy=None,
                                               n_estimators=182, n_jobs=-1,
                                               num_parallel_tree=None, ...),
                       method='isotonic')

In [14]:
calibrated_test_probs = calibrated_production_model.predict_proba(test_df[feature_cols])[:, 1]
calibrated_test_probs = np.clip(calibrated_test_probs, 1e-6, 1 - 1e-6)  # safety clip

calibrated_prediction = pd.DataFrame({
    'client_id': test_df['client_id'],
    'default': calibrated_test_probs,
})
calibrated_prediction.to_csv('prediction_calibrated.csv', index=False)

calibrated_prediction.head()

,client_id,default
0,CC_0012A082B7B7,0.148678
1,CC_0012BC27DFB6,0.162048
2,CC_001563B2143D,0.123839
3,CC_001B46930B6F,0.084110
4,CC_001D771E1C9F,0.063651


# Logistic Regression

Same workflow as the XGBoost sections above: baseline → tune → check
calibration → refit best config on full data → predict on `test.csv`
(both an uncalibrated and a calibrated version).

Logistic regression needs numeric features scaled and categorical features
one-hot encoded (XGBoost/CatBoost can consume raw/categorical columns
directly, sklearn's `LogisticRegression` cannot), so this section adds a
`ColumnTransformer` preprocessing step ahead of the model. Adjust
`categorical_cols` below if your dataset's categorical columns differ from
the UCI "default of credit card clients" schema assumed here.

In [15]:
categorical_cols = [c for c in
    ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
    if c in feature_cols]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
])

## Baseline model

In [16]:
logreg_baseline = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

logreg_baseline.fit(X_train, y_train)

logreg_val_probs = logreg_baseline.predict_proba(X_val)[:, 1]
logreg_baseline_log_loss = log_loss(y_val, logreg_val_probs)
print("Logistic Regression baseline log loss:", logreg_baseline_log_loss)

Logistic Regression baseline log loss: 0.4489287218498791


## Hyperparameter tuning (RandomizedSearchCV)

In [17]:
logreg_param_dist = {
    'model__C': uniform(0.001, 10),
    'model__penalty': ['l1', 'l2'],
    'model__solver': ['saga'],   # saga supports both l1 and l2
}

logreg_search = RandomizedSearchCV(
    Pipeline([
        ('preprocess', preprocessor),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    param_distributions=logreg_param_dist,
    n_iter=30,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='neg_log_loss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

logreg_search.fit(X_train, y_train)

print("Best params:", logreg_search.best_params_)
print("Best CV log loss:", -logreg_search.best_score_)

best_logreg_model = logreg_search.best_estimator_

logreg_tuned_val_probs = best_logreg_model.predict_proba(X_val)[:, 1]
logreg_tuned_log_loss = log_loss(y_val, logreg_tuned_val_probs)
print("Tuned Logistic Regression validation log loss:", logreg_tuned_log_loss)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The ma

KeyboardInterrupt: 

## Calibration check

(There's no early-stopping equivalent for logistic regression — it isn't an
iterative boosting model — so this section goes straight from tuning to
calibration, unlike the XGBoost/CatBoost sections.)

In [ ]:
calibrated_logreg_model = CalibratedClassifierCV(best_logreg_model, method='isotonic', cv=5)
calibrated_logreg_model.fit(X_train, y_train)

logreg_cal_val_probs = calibrated_logreg_model.predict_proba(X_val)[:, 1]
logreg_calibrated_log_loss = log_loss(y_val, logreg_cal_val_probs)
print("Calibrated validation log loss:  ", logreg_calibrated_log_loss)
print("Uncalibrated validation log loss:", logreg_tuned_log_loss)

## Summary of validation results

In [ ]:
logreg_results = pd.DataFrame({
    'model': ['Baseline', 'Tuned (uncalibrated)', 'Tuned + isotonic calibration'],
    'val_log_loss': [logreg_baseline_log_loss, logreg_tuned_log_loss, logreg_calibrated_log_loss],
}).sort_values('val_log_loss').reset_index(drop=True)

logreg_results

## Refit on full training set and predict on test.csv

In [ ]:
# best_logreg_model is a full sklearn Pipeline (preprocessing + model), so it
# can be refit directly on the raw X/y just like the tuned XGBoost model.
logreg_production_model = logreg_search.best_estimator_
logreg_production_model.fit(X, y)

logreg_test_probs = logreg_production_model.predict_proba(test_df[feature_cols])[:, 1]
logreg_test_probs = np.clip(logreg_test_probs, 1e-6, 1 - 1e-6)

logreg_prediction = pd.DataFrame({
    'client_id': test_df['client_id'],
    'default': logreg_test_probs,
})
logreg_prediction.to_csv('prediction_logreg.csv', index=False)

logreg_prediction.head()

## Calibrated version

In [ ]:
calibrated_logreg_production_model = CalibratedClassifierCV(
    Pipeline([
        ('preprocess', preprocessor),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE,
                                      **{k.replace('model__', ''): v
                                         for k, v in logreg_search.best_params_.items()})),
    ]),
    method='isotonic',
    cv=5,
)
calibrated_logreg_production_model.fit(X, y)

logreg_calibrated_test_probs = calibrated_logreg_production_model.predict_proba(test_df[feature_cols])[:, 1]
logreg_calibrated_test_probs = np.clip(logreg_calibrated_test_probs, 1e-6, 1 - 1e-6)

logreg_calibrated_prediction = pd.DataFrame({
    'client_id': test_df['client_id'],
    'default': logreg_calibrated_test_probs,
})
logreg_calibrated_prediction.to_csv('prediction_logreg_calibrated.csv', index=False)

logreg_calibrated_prediction.head()

# CatBoost

Same workflow as the XGBoost sections: baseline → tune → check calibration →
early stopping → refit best config on full data → predict on `test.csv`
(uncalibrated and calibrated versions).

CatBoost handles categorical columns natively via `cat_features` (no
one-hot encoding needed, unlike logistic regression) — reusing the same
`categorical_cols` list defined in the Logistic Regression section above.

## Baseline model

In [18]:
cat_feature_idx = [feature_cols.index(c) for c in categorical_cols]

catboost_baseline = CatBoostClassifier(
    random_state=RANDOM_STATE,
    eval_metric='Logloss',
    cat_features=cat_feature_idx,
    verbose=False,
)

catboost_baseline.fit(X_train, y_train)

catboost_val_probs = catboost_baseline.predict_proba(X_val)[:, 1]
catboost_baseline_log_loss = log_loss(y_val, catboost_val_probs)
print("CatBoost baseline log loss:", catboost_baseline_log_loss)

CatBoost baseline log loss: 0.4347070008762061


## Hyperparameter tuning (RandomizedSearchCV)

In [24]:
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import uniform, randint

catboost_param_dist = {
    'iterations': randint(100, 600),
    'depth': randint(3, 8),
    'learning_rate': uniform(0.01, 0.3),
    'l2_leaf_reg': uniform(1, 10),
    'bagging_temperature': uniform(0, 1),
}

catboost_search = RandomizedSearchCV(
    CatBoostClassifier(
        eval_metric='Logloss',
        random_state=RANDOM_STATE,
        verbose=False
    ),
    param_distributions=catboost_param_dist,
    n_iter=30,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='neg_log_loss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

catboost_search.fit(X_train, y_train)

print("Best params:", catboost_search.best_params_)
print("Best CV log loss:", -catboost_search.best_score_)

best_catboost_model = catboost_search.best_estimator_

catboost_tuned_val_probs = best_catboost_model.predict_proba(X_val)[:, 1]
catboost_tuned_log_loss = log_loss(y_val, catboost_tuned_val_probs)
print("Tuned CatBoost validation log loss:", catboost_tuned_log_loss)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params: {'bagging_temperature': np.float64(0.9488855372533332), 'depth': 6, 'iterations': 113, 'l2_leaf_reg': np.float64(9.08397348116461), 'learning_rate': np.float64(0.10138413075201119)}
Best CV log loss: 0.42513687000613193
Tuned CatBoost validation log loss: 0.43468032782270116


## Calibration check

Checking whether isotonic calibration improves on the tuned model, same as
for XGBoost.

In [28]:
calibrated_catboost_model = CalibratedClassifierCV(best_catboost_model, method='isotonic', cv=5)
calibrated_catboost_model.fit(X_train, y_train)

catboost_cal_val_probs = calibrated_catboost_model.predict_proba(X_val)[:, 1]
catboost_calibrated_log_loss = log_loss(y_val, catboost_cal_val_probs)
print("Calibrated validation log loss:  ", catboost_calibrated_log_loss)
print("Uncalibrated validation log loss:", catboost_tuned_log_loss)

RuntimeError: Cannot clone object CatBoostClassifier(bagging_temperature=np.float64(0.9488855372533332), depth=6, eval_metric='Logloss', iterations=113, l2_leaf_reg=np.float64(9.08397348116461), learning_rate=np.float64(0.10138413075201119), random_state=42, verbose=False), as the constructor either does not set or modifies parameter bagging_temperature

## Early stopping

Same idea as the XGBoost section: give CatBoost a generous cap on
`iterations` and let early stopping decide the actual number of trees,
using the tuned hyperparameters found above (minus `iterations`).

In [25]:
best_catboost_params = dict(catboost_search.best_params_)
best_catboost_params.pop('iterations', None)

final_catboost_model = CatBoostClassifier(
    **best_catboost_params,
    eval_metric='Logloss',
    cat_features=cat_feature_idx,
    iterations=2000,          # high cap; early stopping will cut it short
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    verbose=False,
)

final_catboost_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True,
)

print("Best iteration:", final_catboost_model.get_best_iteration())

Best iteration: 163


In [26]:
final_catboost_val_probs = final_catboost_model.predict_proba(X_val)[:, 1]
final_catboost_log_loss = log_loss(y_val, final_catboost_val_probs)
print("Early-stopped CatBoost validation log loss:", final_catboost_log_loss)

Early-stopped CatBoost validation log loss: 0.43435220082160547


## Summary of validation results

In [27]:
catboost_results = pd.DataFrame({
    'model': ['Baseline', 'Tuned (uncalibrated)', 'Tuned + isotonic calibration', 'Tuned + early stopping'],
    'val_log_loss': [catboost_baseline_log_loss, catboost_tuned_log_loss,
                      catboost_calibrated_log_loss, final_catboost_log_loss],
}).sort_values('val_log_loss').reset_index(drop=True)

catboost_results

NameError: name 'catboost_calibrated_log_loss' is not defined

## Refit best configuration on the full training set and predict on test.csv

Same reasoning as the XGBoost section: early stopping needs a held-out
validation set, so for the full-data refit we fix `iterations` at the
`best_iteration` found above and retrain without early stopping.

In [ ]:
catboost_full_fit_params = dict(best_catboost_params)
catboost_full_fit_params['iterations'] = final_catboost_model.get_best_iteration() + 1

catboost_production_model = CatBoostClassifier(
    **catboost_full_fit_params,
    eval_metric='Logloss',
    cat_features=cat_feature_idx,
    random_state=RANDOM_STATE,
    verbose=False,
)

# Refit on ALL of train.csv (X, y), not just the train split
catboost_production_model.fit(X, y)

In [ ]:
catboost_test_probs = catboost_production_model.predict_proba(test_df[feature_cols])[:, 1]
catboost_test_probs = np.clip(catboost_test_probs, 1e-6, 1 - 1e-6)

catboost_prediction = pd.DataFrame({
    'client_id': test_df['client_id'],
    'default': catboost_test_probs,
})
catboost_prediction.to_csv('prediction_catboost.csv', index=False)

catboost_prediction.head()

## Calibrated version

In [ ]:
calibrated_catboost_production_model = CalibratedClassifierCV(
    CatBoostClassifier(
        **catboost_full_fit_params,
        eval_metric='Logloss',
        cat_features=cat_feature_idx,
        random_state=RANDOM_STATE,
        verbose=False,
    ),
    method='isotonic',
    cv=5,
)
calibrated_catboost_production_model.fit(X, y)

catboost_calibrated_test_probs = calibrated_catboost_production_model.predict_proba(test_df[feature_cols])[:, 1]
catboost_calibrated_test_probs = np.clip(catboost_calibrated_test_probs, 1e-6, 1 - 1e-6)

catboost_calibrated_prediction = pd.DataFrame({
    'client_id': test_df['client_id'],
    'default': catboost_calibrated_test_probs,
})
catboost_calibrated_prediction.to_csv('prediction_catboost_calibrated.csv', index=False)

catboost_calibrated_prediction.head()